# BETO multitarea — encoder compartido
Notebook nuevo e independiente del baseline plano y del jerárquico.
Activa GPU en Colab y ejecuta todas las celdas. Sube manualmente
`train-experimental.jsonl` (758) y `dev.jsonl` (160), campos `id`, `text`, `label`.
Puedes subir también `flat_beto_results.json` y `hierarchical_beto_results.json`.

Primero usa `RUN_SMOKE_TEST = True`; después cambia a `False` y ejecuta las celdas
desde configuración para las tres semillas. El smoke test no produce resultados científicos.
Un encoder BETO base, dos cabezas, pérdida 1:1 y soft gating. Solo macro-F1 final DEV
selecciona el checkpoint; DEV expuesto no constituye prueba independiente ni gold experto.
No se accede a V/S. No se ejecutó entrenamiento durante la preparación del notebook.
Base y tokenizer fijados a la revisión y hashes comprobados del baseline y jerárquico.
El baseline usa padding fijo y este encargo exige padding dinámico: queda registrado.


In [ ]:
# 2. Instalación de dependencias (antes de importar transformers)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.6', 'huggingface-hub==0.36.2', 'tokenizers==0.22.2',
    'safetensors==0.8.0', 'numpy', 'pandas', 'scikit-learn', 'matplotlib', 'tqdm'], check=True)
# BERT es texto: evitar conflictos de extensiones opcionales de audio/visión de Colab.
import transformers.utils.import_utils as hf_imports
hf_imports._torchvision_available = False
hf_imports._librosa_available = False


In [ ]:
# 3. Imports
import os, gc, json, math, random, hashlib, shutil, platform
from pathlib import Path
from collections import Counter
from itertools import islice
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import files
from transformers import AutoTokenizer, AutoModel, DataCollatorWithPadding, set_seed
from huggingface_hub import snapshot_download
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix

def save_json(path, value):
    path=Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temp=path.with_suffix(path.suffix+'.tmp')
    temp.write_text(json.dumps(value,ensure_ascii=False,indent=2,allow_nan=False)+'\n',encoding='utf-8')
    temp.replace(path)

def digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()


In [ ]:
# Configuración fija de un encoder compartido por semilla
MODEL_NAME = 'dccuchile/bert-base-spanish-wwm-cased'
MODEL_REVISION = 'c4d86612f51b4f46759c8390d1798c2febe71b93'
MAX_LENGTH = 384
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
MAX_EPOCHS = 8
PATIENCE = 2
MIN_DELTA = 1e-4
WARMUP_RATIO = 0.10
GRAD_CLIP = 1.0
MICRO_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
SEEDS = [42, 123, 2026]
DROPOUT = 0.1
# Permitir recuperación normal antes de declarar persistencia: 20 omisiones consecutivas.
MAX_CONSECUTIVE_AMP_SKIPS = 20
RESULTS = Path('/content/results')
CHECKPOINTS = Path('/content/checkpoints')
BASE_HASHES = {
 'config.json':'3694a761b0ba882c24baab95df01cf7e7e1424797af557272e944ec2452f9f31',
 'pytorch_model.bin':'e131a95091c777bbd45250fb647fec415010cb8cd1ad6e1d59babeb82a0be360',
 'special_tokens_map.json':'bd6ed009009f8264d0ef87d5b50798cb57c5219a0bbb7c8973372855241aa05f',
 'tokenizer.json':'ea7a58026720ba45a8a401d9f86bbe8337f2725d3b02d3a1011305c46cbbd9cd',
 'tokenizer_config.json':'2f5cde6bbb9959fccd280e9e9c08d41d05e96743fac1a3b5b67e0e5691ae7d10',
 'vocab.txt':'b8f1c939e21273bd19cd885d0b6d7eb11244240ef81f62e30dcf84b4c970ce36'}

RUN_SMOKE_TEST = True


In [ ]:
# 5. GPU y reproducibilidad
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
os.environ['TOKENIZERS_PARALLELISM']='false'
if not torch.cuda.is_available():raise RuntimeError('Activa GPU en Entorno de ejecución > Cambiar tipo.')
if tuple(int(v) for v in torch.__version__.split('+')[0].split('.')[:2]) < (2,6):
    raise RuntimeError('Se requiere PyTorch >=2.6 para cargar el BETO base .bin; usa un runtime Colab actualizado.')
DEVICE=torch.device('cuda')
print('GPU:',torch.cuda.get_device_name(),'PyTorch:',torch.__version__,'CUDA:',torch.version.cuda)
torch.backends.cudnn.benchmark=False
torch.backends.cudnn.deterministic=True
torch.backends.cuda.matmul.allow_tf32=False
torch.backends.cudnn.allow_tf32=False
torch.use_deterministic_algorithms(True)

def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed);set_seed(seed)

def release_gpu():
    gc.collect();torch.cuda.empty_cache()

seed_all(SEEDS[0])
print('Semillas fijas; se registra el entorno. GPU/versiones distintas pueden producir diferencias numéricas.')


In [ ]:
# 6. Carga manual TRAIN/DEV. Selecciona ambos archivos en el cuadro de carga.
uploaded = files.upload()
def detect_upload(wanted):
    matches=[name for name in uploaded if Path(name).name == wanted]
    if len(matches)!=1:raise ValueError(f'Carga exactamente un archivo llamado {wanted}; recibidos: {list(uploaded)}')
    return matches[0]
TRAIN_FILE=detect_upload('train-experimental.jsonl')
DEV_FILE=detect_upload('dev.jsonl')
def read_jsonl(name):
    records=[]
    for line_number,line in enumerate(uploaded[name].decode('utf-8-sig').splitlines(),1):
        if not line.strip():continue
        try:record=json.loads(line)
        except Exception as exc:raise ValueError(f'{name}, línea {line_number}: JSON inválido') from exc
        if not isinstance(record,dict):raise ValueError(f'{name}, línea {line_number}: se espera objeto JSON')
        records.append(record)
    return records
train_rows=read_jsonl(TRAIN_FILE)
dev_rows=read_jsonl(DEV_FILE)
input_hashes={'train':hashlib.sha256(uploaded[TRAIN_FILE]).hexdigest(),
              'dev':hashlib.sha256(uploaded[DEV_FILE]).hexdigest()}
# Una carga opcional en este mismo cuadro también se reconoce más adelante.
flat_uploaded=uploaded.get('flat_beto_results.json')
print('TRAIN:',len(train_rows),'DEV:',len(dev_rows))


In [ ]:
# 7. Validaciones críticas. Los metadatos se conservan; nunca son features.
EXPECTED_LABELS=(
 'campanias_conflictos_militares','contexto_colonial_antecedentes','crisis_ideas_emancipadoras',
 'liderazgos_diplomacia_proyectos','no_relevante','organizacion_consecuencias_republicanas','participacion_social_regional')
def text_hash(r):return hashlib.sha256(r['text'].encode('utf-8')).hexdigest()
def validate_split(rows,name,expected):
    if len(rows)!=expected:raise ValueError(f'{name}: esperado {expected}, recibido {len(rows)}')
    for i,r in enumerate(rows):
        for key in ('id','text','label'):
            if not isinstance(r.get(key),str) or not r[key].strip():raise ValueError(f'{name}[{i}]: falta {key} no vacío de tipo string')
        if r['label'] not in EXPECTED_LABELS:raise ValueError(f'{name}: label desconocido {r["label"]!r}')
        if r.get('text_sha256') is not None and r['text_sha256']!=text_hash(r):raise ValueError(f'{name}: hash de texto incorrecto {r["id"]}')
    if len({r['id'] for r in rows})!=len(rows):raise ValueError(f'{name}: IDs duplicados')
    if len({text_hash(r) for r in rows})!=len(rows):raise ValueError(f'{name}: textos exactamente duplicados')
    if set(r['label'] for r in rows)!=set(EXPECTED_LABELS):raise ValueError(f'{name}: faltan clases esperadas')
validate_split(train_rows,'TRAIN',758);validate_split(dev_rows,'DEV',160)
for key,fn in [('id',lambda r:r['id']),('text_hash',text_hash)]:
    if {fn(r) for r in train_rows}&{fn(r) for r in dev_rows}:raise ValueError(f'Solapamiento TRAIN/DEV: {key}')
for key in ('family','component','source_id'):
    a={str(r[key]) for r in train_rows if r.get(key) is not None and str(r[key]).strip()}
    b={str(r[key]) for r in dev_rows if r.get(key) is not None and str(r[key]).strip()}
    if a&b:raise ValueError(f'Solapamiento TRAIN/DEV en {key}: {sorted(a&b)}')
    if not all(r.get(key) for r in train_rows+dev_rows):print(f'{key}: comprobación parcial; metadatos ausentes en algunas filas.')
print('Validaciones críticas correctas. No se creó ni modificó ninguna partición.')


In [ ]:
# 9. Mappings explícitos e inmutables
FINAL_LABELS=list(EXPECTED_LABELS)
A_LABELS=['no_relevante','relevante']
B_LABELS=['contexto_colonial_antecedentes','crisis_ideas_emancipadoras','campanias_conflictos_militares',
          'liderazgos_diplomacia_proyectos','organizacion_consecuencias_republicanas','participacion_social_regional']
FINAL_TO_ID={l:i for i,l in enumerate(FINAL_LABELS)}
B_TO_ID={l:i for i,l in enumerate(B_LABELS)}
NR_ID=FINAL_TO_ID['no_relevante']
assert NR_ID==4 and set(B_LABELS)==set(FINAL_LABELS)-{'no_relevante'}
assert [FINAL_TO_ID[l] for l in B_LABELS]==[1,2,0,3,5,6]
print('Orden final:',FINAL_TO_ID)


In [ ]:
# 8. Distribución de clases
distribution=pd.DataFrame({'TRAIN':Counter(r['label'] for r in train_rows),'DEV':Counter(r['label'] for r in dev_rows)}).reindex(EXPECTED_LABELS)
display(distribution)


In [ ]:
# 10. Tokenización única, padding dinámico, sin alterar texto
BASE_PATH=Path(snapshot_download(MODEL_NAME,revision=MODEL_REVISION,allow_patterns=list(BASE_HASHES)))
for name,expected in BASE_HASHES.items():
    if digest(BASE_PATH/name)!=expected:raise ValueError(f'BETO base/tokenizer cambiado: {name}')
tokenizer=AutoTokenizer.from_pretrained(BASE_PATH,use_fast=True,local_files_only=True)
collator=DataCollatorWithPadding(tokenizer,padding=True,return_tensors='pt')
def tokenize_rows(rows):
    enc=tokenizer([r['text'] for r in rows],truncation=True,max_length=MAX_LENGTH,padding=False)
    return [{k:enc[k][i] for k in enc} for i in range(len(rows))]
train_tokens=tokenize_rows(train_rows);dev_tokens=tokenize_rows(dev_rows)
for name,tokens in [('TRAIN',train_tokens),('DEV',dev_tokens)]:
    print(name,'textos con longitud tokenizada ==384:',sum(len(t['input_ids'])==MAX_LENGTH for t in tokens))
print('Alcanzar 384 no distingue por sí solo longitud exacta de truncamiento.')

def make_loader(rows,tokens,shuffle,seed,indices=None):
    if indices is None: indices=range(len(rows))
    items=[{**tokens[i], 'labels':FINAL_TO_ID[rows[i]['label']]} for i in indices]
    return torch.utils.data.DataLoader(items,batch_size=MICRO_BATCH_SIZE,shuffle=shuffle,
        generator=torch.Generator().manual_seed(seed),num_workers=0,collate_fn=collator)

train_y=torch.tensor([FINAL_TO_ID[r['label']] for r in train_rows],device=DEVICE)
FINAL_TO_B=torch.tensor([B_TO_ID.get(l,-100) for l in FINAL_LABELS],device=DEVICE)
def balanced_weights(y,k):
    counts=torch.bincount(y,minlength=k)
    assert bool((counts>0).all())
    return len(y)/(k*counts.float())
weights_A=balanced_weights((train_y!=NR_ID).long(),2)
weights_B=balanced_weights(FINAL_TO_B[train_y[train_y!=NR_ID]],6)
print('Pesos A:',weights_A.tolist(),'Pesos B:',weights_B.tolist())


In [ ]:
# Un encoder y dos cabezas sobre el mismo CLS
class MultitaskBETO(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder=AutoModel.from_pretrained(BASE_PATH,local_files_only=True,use_safetensors=False,add_pooling_layer=False)
        h=self.encoder.config.hidden_size
        self.head_a=torch.nn.Sequential(torch.nn.Dropout(0.1),torch.nn.Linear(h,2))
        self.head_b=torch.nn.Sequential(torch.nn.Dropout(0.1),torch.nn.Linear(h,6))
        self.encoder.gradient_checkpointing_enable()
    def forward(self,**inputs):
        cls=self.encoder(**inputs).last_hidden_state[:,0,:]
        return self.head_a(cls),self.head_b(cls)

def loss_numerators(a,b,y):
    mask=y!=NR_ID  # Exclusivamente referencia real; nunca predicción de A.
    rel=torch.nn.functional.cross_entropy(a.float(),mask.long(),weight=weights_A,reduction='sum')
    hist=(torch.nn.functional.cross_entropy(b[mask].float(),FINAL_TO_B[y[mask]],
           weight=weights_B,reduction='sum') if bool(mask.any()) else b.float().sum()*0.0)
    return rel,hist

def loss_denominators(y):
    mask=y!=NR_ID
    return weights_A[mask.long()].sum(),weights_B[FINAL_TO_B[y[mask]]].sum()


In [ ]:
# 11. Métricas y evaluación en FP32
def metrics(y,probabilities,labels):
    p=np.asarray(probabilities,dtype=np.float64);y=np.asarray(y,dtype=int)
    assert p.shape==(len(y),len(labels)) and np.isfinite(p).all()
    assert (p>=0).all() and np.allclose(p.sum(1),1,atol=1e-6)
    pred=p.argmax(1)
    precision,recall,f1,support=precision_recall_fscore_support(y,pred,labels=np.arange(len(labels)),zero_division=0)
    return {'macro_f1':float(f1.mean()),'accuracy':float(accuracy_score(y,pred)),
      'precision_macro':float(precision.mean()),'recall_macro':float(recall.mean()),
      'loss':float(-np.log(np.maximum(p[np.arange(len(y)),y],1e-15)).mean()),
      'per_class':[{'label':l,'precision':float(precision[i]),'recall':float(recall[i]),'f1':float(f1[i]),'support':int(support[i])} for i,l in enumerate(labels)],
      'confusion_matrix':confusion_matrix(y,pred,labels=np.arange(len(labels))).tolist()}

# 16. Soft gating: todas las filas pasan por A y B; no umbrales
def soft_gate(pa,pb):
    pa=np.asarray(pa,dtype=np.float64);pb=np.asarray(pb,dtype=np.float64)
    assert pa.ndim==pb.ndim==2 and pa.shape[1]==2 and pb.shape==(len(pa),6)
    assert np.isfinite(pa).all() and np.isfinite(pb).all() and (pa>=0).all() and (pb>=0).all()
    assert np.allclose(pa.sum(1),1,atol=1e-6) and np.allclose(pb.sum(1),1,atol=1e-6)
    final=np.zeros((len(pa),7),dtype=np.float64)
    final[:,NR_ID]=pa[:,0]
    for j,label in enumerate(B_LABELS):final[:,FINAL_TO_ID[label]]=pa[:,1]*pb[:,j]
    assert np.allclose(final.sum(1),1,atol=2e-6)
    return final

def error_counts(y,pred):
    y=np.asarray(y);pred=np.asarray(pred)
    result={'relevance_to_NR_errors':int(((y!=NR_ID)&(pred==NR_ID)).sum()),
            'NR_to_relevance_errors':int(((y==NR_ID)&(pred!=NR_ID)).sum()),
            'historical_internal_errors':int(((y!=NR_ID)&(pred!=NR_ID)&(y!=pred)).sum())}
    assert sum(result.values())==int((y!=pred).sum())
    return result


In [ ]:
def evaluate(model,loader):
    model.eval(); aa=[];bb=[];yy=[]
    with torch.inference_mode():
        for batch in loader:
            yy.extend(batch['labels'].tolist())
            a,b=model(**{k:v.to(DEVICE) for k,v in batch.items() if k!='labels'})
            aa.extend(a.float().softmax(-1).cpu().tolist())
            bb.extend(b.float().softmax(-1).cpu().tolist())
    pa=np.asarray(aa);pb=np.asarray(bb);y=np.asarray(yy);p=soft_gate(pa,pb)
    mask=y!=NR_ID
    hist_y=np.asarray([B_TO_ID[FINAL_LABELS[i]] for i in y[mask]])
    return {'metrics':metrics(y,p,FINAL_LABELS),
            'head_a':metrics(mask.astype(int),pa,A_LABELS),
            'head_b':metrics(hist_y,pb[mask],B_LABELS),
            'errors':error_counts(y,p.argmax(1))},p,pa,pb

class EarlyStopping:
    def __init__(self):self.best=-math.inf;self.bad=0;self.best_epoch=None
    def update(self,score,epoch):
        if not math.isfinite(score):raise RuntimeError('Macro-F1 no finito')
        previous=self.best
        improved=score>previous
        if improved:self.best=score;self.best_epoch=epoch
        if score>previous+MIN_DELTA:self.bad=0
        else:self.bad+=1
        return improved,self.bad>=PATIENCE


In [ ]:
def optimizer_tools(model,loader):
    optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    total=MAX_EPOCHS*math.ceil(len(loader)/GRADIENT_ACCUMULATION_STEPS)
    warmup=math.floor(WARMUP_RATIO*total)
    def factor(step):
        if warmup and step<warmup:return step/warmup
        return max(0.,(total-step)/max(1,total-warmup))
    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,factor)
    scaler=torch.amp.GradScaler('cuda')
    return optimizer,scheduler,scaler


def train_epoch(model,loader,optimizer,scheduler,scaler,state,emit,max_attempts=None):
    model.train();optimizer.zero_grad(set_to_none=True)
    sums=np.zeros(4);updates=skips=attempts=0
    iterator=iter(loader)
    while group:=list(islice(iterator,GRADIENT_ACCUMULATION_STEPS)):
        # Normalización por cabeza en el batch efectivo completo, incluida la cola.
        target=torch.cat([batch['labels'] for batch in group]).to(DEVICE)
        da,db=loss_denominators(target)
        group_finite=True
        for batch in group:
            y=batch['labels'].to(DEVICE)
            with torch.autocast('cuda',dtype=torch.float16):
                a,b=model(**{k:v.to(DEVICE) for k,v in batch.items() if k!='labels'})
            na,nb=loss_numerators(a,b,y)
            loss=na/da+(nb/db if bool(db>0) else nb)
            group_finite=group_finite and bool(torch.isfinite(loss))
            scaler.scale(loss).backward()
            if bool(torch.isfinite(na)) and bool(torch.isfinite(nb)):
                ma,mb=loss_denominators(y)
                sums+=np.array([na.detach().item(),nb.detach().item(),ma.item(),mb.item()])
        scaler.unscale_(optimizer)
        params=[p for p in model.parameters() if p.grad is not None]
        finite=all(bool(torch.isfinite(p.grad).all()) for p in params)
        if finite:
            norm=torch.linalg.vector_norm(torch.stack([torch.linalg.vector_norm(p.grad.double()) for p in params]))
            factor=min(1.,GRAD_CLIP/(float(norm)+1e-6))
            for p in params:p.grad.mul_(factor)
        before=scaler.get_scale()
        scaler.step(optimizer);scaler.update()
        skipped=scaler.get_scale()<before
        optimizer.zero_grad(set_to_none=True)
        attempts+=1
        if skipped:skips+=1;state['consecutive_skips']+=1
        else:updates+=1;state['consecutive_skips']=0;scheduler.step()
        emit({'attempt':attempts,'skipped':skipped,'finite_loss':group_finite,
              'finite_gradients':finite,'scale_before':before,'scale_after':scaler.get_scale()})
        if not group_finite and not skipped:raise RuntimeError('Loss no finita sin recuperación AMP')
        if state['consecutive_skips']>=MAX_CONSECUTIVE_AMP_SKIPS:
            raise RuntimeError('NaN/Inf persistentes tras 20 omisiones AMP consecutivas')
        if max_attempts is not None and attempts>=max_attempts:break
    la=sums[0]/sums[2] if sums[2] else 0.
    lb=sums[1]/sums[3] if sums[3] else 0.
    return {'train_total_loss':la+lb,'train_relevance_loss':la,'train_historical_loss':lb,
            'updates':updates,'amp_skips':skips,'learning_rate':optimizer.param_groups[0]['lr']}


In [ ]:
# Protocolo y protección de ejecuciones anteriores
protocol={'model':MODEL_NAME,'revision':MODEL_REVISION,'labels':FINAL_LABELS,'head_a':A_LABELS,
 'head_b':B_LABELS,'input_hashes':input_hashes,'train':758,'dev':160,'seeds':SEEDS,
 'max_length':MAX_LENGTH,'padding':'dynamic','lr':LEARNING_RATE,'weight_decay':WEIGHT_DECAY,
 'max_epochs':MAX_EPOCHS,'patience':PATIENCE,'min_delta':MIN_DELTA,'warmup_ratio':WARMUP_RATIO,
 'micro_batch':MICRO_BATCH_SIZE,'accumulation':GRADIENT_ACCUMULATION_STEPS,'grad_clip':GRAD_CLIP,
 'loss_weights':[1,1],'weights_A':weights_A.tolist(),'weights_B':weights_B.tolist(),
 'selection':'strict maximum DEV macro-F1 final 7; ties earliest; patience against previous best',
 'python':platform.python_version(),'torch':torch.__version__,'cuda':torch.version.cuda,
 'gpu':torch.cuda.get_device_name(),'expert_gold':False,'DEV_previously_exposed':True}

def run_seed(seed):
    seed_all(seed)
    loader=make_loader(train_rows,train_tokens,True,seed)
    dev_loader=make_loader(dev_rows,dev_tokens,False,seed)
    destination=CHECKPOINTS/f'seed_{seed}'
    destination.mkdir(parents=True,exist_ok=False)
    model=MultitaskBETO().to(DEVICE)
    optimizer,scheduler,scaler=optimizer_tools(model,loader)
    history=[];stopper=EarlyStopping();state={'consecutive_skips':0}
    try:
        with (RESULTS/f'amp_seed_{seed}.jsonl').open('w',encoding='utf-8') as events:
            for epoch in range(1,MAX_EPOCHS+1):
                def emit(event):
                    events.write(json.dumps({'seed':seed,'epoch':epoch,**event})+'\n');events.flush()
                    if event['skipped']:print('Recuperación AMP:',event)
                train=train_epoch(model,loader,optimizer,scheduler,scaler,state,emit)
                info,p,pa,pb=evaluate(model,dev_loader)
                score=info['metrics']['macro_f1']
                improved,stop=stopper.update(score,epoch)
                history.append({'epoch':epoch,**train,'dev_macro_f1_7':score,
                    'dev_relevance_f1':info['head_a']['macro_f1'],'dev_historical_f1':info['head_b']['macro_f1']})
                pd.DataFrame(history).to_csv(RESULTS/f'multitask_history_seed_{seed}.csv',index=False)
                if improved:
                    torch.save(model.state_dict(),destination/'model.pt')
                    model.encoder.config.save_pretrained(destination)
                    tokenizer.save_pretrained(destination)
                    save_json(destination/'multitask_config.json',{'protocol':protocol,'seed':seed,'epoch':epoch,
                        'architecture':'AutoModel CLS -> Dropout(.1)/Linear(2), Dropout(.1)/Linear(6)'})
                    best_p=p.copy()
                print('seed',seed,'epoch',epoch,'DEV macro-F1 final',score)
                if stop:break
        if sum(h['updates'] for h in history)==0:raise RuntimeError('Cero updates efectivos')
        model.load_state_dict(torch.load(destination/'model.pt',map_location='cpu',weights_only=True))
        info,p,pa,pb=evaluate(model,dev_loader)
        assert np.allclose(p,best_p,atol=1e-6,rtol=0),'Recarga no reproduce checkpoint'
        pred=p.argmax(1)
        with (RESULTS/f'multitask_seed_{seed}_predictions.jsonl').open('w',encoding='utf-8') as f:
            for i,r in enumerate(dev_rows):
                record={k:r[k] for k in ('family','component','source_id') if k in r}
                record.update(id=r['id'],text_sha256=text_hash(r),reference=r['label'],prediction=FINAL_LABELS[pred[i]],
                  probabilities_7=p[i].tolist(),p_relevant=float(pa[i,1]),p_not_relevant=float(pa[i,0]),
                  probabilities_history_6=pb[i].tolist(),seed=seed)
                f.write(json.dumps(record,ensure_ascii=False,allow_nan=False)+'\n')
        info.update(seed=seed,selected_epoch=stopper.best_epoch,history=history)
        save_json(RESULTS/f'multitask_seed_{seed}_metrics.json',info)
        for name,labels,m in [('confusion',FINAL_LABELS,info['metrics']),('head_a_confusion',A_LABELS,info['head_a']),
                              ('head_b_confusion',B_LABELS,info['head_b'])]:
            pd.DataFrame(m['confusion_matrix'],index=labels,columns=labels).to_csv(RESULTS/f'multitask_{name}_seed_{seed}.csv')
        return info
    finally:
        del model,optimizer,scheduler,scaler
        release_gpu()


In [ ]:
# Smoke: datos reales, sin guardar checkpoints ni métricas científicas
def smoke_update(model,loader,optimizer,scheduler,scaler):
    state={'consecutive_skips':0}
    for attempt in range(1,MAX_CONSECUTIVE_AMP_SKIPS+1):
        before=scheduler.last_epoch
        def emit(event):print({'smoke_attempt':attempt,**event})
        result=train_epoch(model,loader,optimizer,scheduler,scaler,state,emit,max_attempts=1)
        assert scheduler.last_epoch-before==result['updates']
        if result['updates']==1:return result
        assert result['amp_skips']==1,'Smoke sin update ni omisión AMP registrada'
        # Conserva modelo, optimizer y GradScaler: la escala reducida se usa en el siguiente intento.
    raise RuntimeError('Smoke: AMP no se recuperó dentro del límite de persistencia')

def run_smoke_test():
    seed_all(42)
    nr=next(i for i,r in enumerate(train_rows) if r['label']=='no_relevante')
    rel=next(i for i,r in enumerate(train_rows) if r['label']!='no_relevante')
    loader=make_loader(train_rows,train_tokens,False,42,[nr,rel])
    model=MultitaskBETO().to(DEVICE)
    optimizer,scheduler,scaler=optimizer_tools(model,loader)
    try:
        batch=next(iter(loader));y=batch['labels'].to(DEVICE)
        a,b=model(**{k:v.to(DEVICE) for k,v in batch.items() if k!='labels'})
        assert a.shape==(2,2) and b.shape==(2,6)
        na,nb=loss_numerators(a,b,y)
        grad=torch.autograd.grad(nb,b,retain_graph=True)[0]
        assert bool((grad[y==NR_ID]==0).all()) and bool((grad[y!=NR_ID]!=0).any())
        _,zero=loss_numerators(a[:1],b[:1],y[:1]);assert zero.item()==0
        assert bool(torch.isfinite(na+nb))
        smoke_update(model,loader,optimizer,scheduler,scaler)
        # Evaluación completa para incluir las seis referencias históricas.
        info,p,pa,pb=evaluate(model,make_loader(dev_rows,dev_tokens,False,42))
        assert p.shape==(160,7) and np.allclose(p.sum(1),1,atol=2e-6)
        print('Smoke aprobado; no es resultado científico.')
    finally:
        del model,optimizer,scheduler,scaler
        release_gpu()

results_by_seed=[]
if RUN_SMOKE_TEST:
    run_smoke_test()
else:
    if RESULTS.exists() or CHECKPOINTS.exists():
        raise RuntimeError('results/checkpoints ya existen. Descarga la ejecución anterior y usa una sesión Colab nueva.')
    RESULTS.mkdir();CHECKPOINTS.mkdir()
    save_json(RESULTS/'protocol.json',protocol)
    (RESULTS/'environment.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True),encoding='utf-8')
    try:
        for seed in SEEDS:results_by_seed.append(run_seed(seed))
    except Exception as exc:
        save_json(RESULTS/'failure.json',{'error':repr(exc),'completed_seeds':[r['seed'] for r in results_by_seed]})
        raise


In [ ]:
complete=not RUN_SMOKE_TEST and [r['seed'] for r in results_by_seed]==SEEDS
if complete:
    metrics_table=pd.DataFrame([{'seed':r['seed'],'selected_epoch':r['selected_epoch'],
       'macro_f1':r['metrics']['macro_f1'],'accuracy':r['metrics']['accuracy'],
       'head_a_macro_f1':r['head_a']['macro_f1'],'head_b_macro_f1':r['head_b']['macro_f1'],**r['errors']}
       for r in results_by_seed])
    per_class_table=pd.DataFrame([{'seed':r['seed'],**c} for r in results_by_seed for c in r['metrics']['per_class']])
    metrics_table.to_csv(RESULTS/'multitask_metrics_by_seed.csv',index=False)
    per_class_table.to_csv(RESULTS/'multitask_per_class.csv',index=False)
    summary={'results':results_by_seed,'protocol':protocol,'labels':FINAL_LABELS,
       'macro_f1_mean':float(metrics_table.macro_f1.mean()),'macro_f1_std':float(metrics_table.macro_f1.std(ddof=1)),
       'accuracy_mean':float(metrics_table.accuracy.mean()),'std_ddof':1}
    save_json(RESULTS/'multitask_summary.json',summary)
    display(metrics_table);display(per_class_table)
    print('Macro-F1 media:',summary['macro_f1_mean'],'Std muestral:',summary['macro_f1_std'])
else:
    print('Sin resumen científico: modo smoke o ejecución incompleta.')


In [ ]:
if complete:
    for r in results_by_seed:
        seed=r['seed'];h=pd.DataFrame(r['history'])
        fig,axes=plt.subplots(1,2,figsize=(12,4))
        h.plot(x='epoch',y='dev_macro_f1_7',ax=axes[0],title=f'Seed {seed}: DEV final')
        h.plot(x='epoch',y=['train_total_loss','train_relevance_loss','train_historical_loss'],ax=axes[1])
        fig.tight_layout();fig.savefig(RESULTS/f'curves_seed_{seed}.png',dpi=160);plt.show()
        fig,ax=plt.subplots(figsize=(7,6));cm=np.asarray(r['metrics']['confusion_matrix'])
        ax.imshow(cm,cmap='Blues');short=['MIL','COL','IDE','LID','NR','REP','SOC']
        ax.set(xticks=range(7),yticks=range(7),xticklabels=short,yticklabels=short,
               xlabel='Predicción',ylabel='Referencia',title=f'Seed {seed}')
        for i in range(7):
            for j in range(7):ax.text(j,i,str(cm[i,j]),ha='center',va='center',color='white' if cm[i,j]>cm.max()/2 else 'black')
        fig.tight_layout();fig.savefig(RESULTS/f'confusion_seed_{seed}.png',dpi=160);plt.show()
    ax=per_class_table.pivot(index='label',columns='seed',values='f1').reindex(FINAL_LABELS).plot.bar(figsize=(12,6),ylim=(0,1),ylabel='F1')
    ax.figure.tight_layout();ax.figure.savefig(RESULTS/'f1_per_class.png',dpi=160);plt.show()


In [ ]:
# Opcional: subir ahora resultados anteriores, o incluirlos en la carga inicial.
# JSON: {"results":[{"seed":42,"metrics":{"macro_f1":0.5,"accuracy":0.5}}, ...]}
# También admite lista de filas {seed,macro_f1,accuracy}. Deben estar las tres seeds.
UPLOAD_COMPARISON = False
if complete:
    optional=dict(uploaded)
    if UPLOAD_COMPARISON:optional.update(files.upload())
    tables={'multitarea':metrics_table[['seed','macro_f1','accuracy']]}
    for name,filename in [('plano','flat_beto_results.json'),('jerárquico','hierarchical_beto_results.json')]:
        if filename not in optional:continue
        obj=json.loads(optional[filename].decode('utf-8-sig'))
        rows=obj if isinstance(obj,list) else obj['results']
        rows=[{'seed':int(r['seed']),'macro_f1':float(r.get('metrics',r)['macro_f1']),
               'accuracy':float(r.get('metrics',r)['accuracy'])} for r in rows]
        table=pd.DataFrame(rows)
        assert len(table)==3 and set(table.seed)==set(SEEDS) and not table.seed.duplicated().any()
        assert np.isfinite(table[['macro_f1','accuracy']]).all().all()
        assert table[['macro_f1','accuracy']].ge(0).all().all() and table[['macro_f1','accuracy']].le(1).all().all()
        if isinstance(obj,dict):
            assert obj.get('labels',FINAL_LABELS)==FINAL_LABELS
            hashes=obj.get('protocol',{}).get('input_hashes')
            if hashes is not None:assert hashes==input_hashes,'Diferentes datos TRAIN/DEV'
        tables[name]=table
    if len(tables)>1:
        print('Comparación descriptiva; sin hashes compatibles no se acredita automáticamente identidad de datos.')
        comparison=pd.DataFrame({name:t.set_index('seed').macro_f1 for name,t in tables.items()}).reindex(SEEDS)
        models=pd.DataFrame([{'Modelo':name,'Macro-F1 medio':t.macro_f1.mean(),'Std':t.macro_f1.std(ddof=1),
                             'Accuracy media':t.accuracy.mean()} for name,t in tables.items()])
        display(models);display(comparison.reindex(columns=['plano','jerárquico','multitarea']))
        models.to_csv(RESULTS/'comparison_models.csv',index=False);comparison.to_csv(RESULTS/'comparison_by_seed.csv')
        ax=comparison.plot.bar(rot=0,ylabel='Macro-F1 DEV',ylim=(0,1))
        ax.figure.tight_layout();ax.figure.savefig(RESULTS/'comparison.png',dpi=160);plt.show()


In [ ]:
DOWNLOAD_CHECKPOINTS = False
if complete:
    files.download(shutil.make_archive('/content/multitask_results','zip',RESULTS))
    if DOWNLOAD_CHECKPOINTS:
        files.download(shutil.make_archive('/content/multitask_checkpoints','zip',CHECKPOINTS))
